# SensorFusion-HAR ESP32 V2 Useful-11 Final\n
\n
Final selected model: `sensorfusion_esp32_v2_useful11_final`.\n
\n
- Accuracy: `0.8935640138`\n
- Macro F1: `0.9027877942`\n
- Minimum per-class F1: `0.8342412451`\n
- Input: `(1, 50, 6)` acc+gyro window\n
- Classes: Walking, Sitting, Standing, Lying Down, Stairs Up, Stairs Down, Jogging, Jumping, Cycling, Running, Waist Bending\n

## 1. Colab setup\n
Run this cell in Colab/Python 3.10-3.11 if you want to regenerate TFLite artifacts. Local Python 3.14 can use the checkpoint/ONNX artifacts but usually cannot install the full TensorFlow conversion stack cleanly.

In [ ]:
# Optional Colab setup:\n
# !pip install -q numpy scipy scikit-learn matplotlib onnx onnx2tf tensorflow ai-edge-litert\n
# After installing TensorFlow/onnx2tf in Colab, restart the runtime if requested.\n

## 2. Inspect final artifacts

In [ ]:
from pathlib import Path\n
import json\n
\n
ROOT = Path('.')\n
CKPT = ROOT / 'checkpoints_v2' / 'best_sensorfusion_esp32_v2_useful11_final.pt'\n
ONNX = ROOT / 'exports' / 'esp32_v2' / 'sensorfusion_esp32_v2_useful11_final.onnx'\n
CALIB = ROOT / 'exports' / 'esp32_v2' / 'calib_data_v2.npy'\n
SUMMARY = ROOT / 'outputs' / 'esp32_v2' / 'summary_v2_useful11_final.json'\n
MANIFEST = ROOT / 'outputs' / 'esp32_v2' / 'dataset_manifest_v2.json'\n
\n
for p in [CKPT, ONNX, CALIB, SUMMARY, MANIFEST]:\n
    print(p, 'OK' if p.exists() else 'MISSING', f'{p.stat().st_size/1024:.1f} KB' if p.exists() else '')\n
\n
summary = json.loads(SUMMARY.read_text())\n
print('accuracy:', summary['accuracy'])\n
print('macro_f1:', summary['macro_f1'])\n
print('min_f1:', summary['min_f1'])\n
summary['per_class_f1']\n

## 3. Optional local verification

In [ ]:
# Run from the repository root if tests are included:\n
# !python -m pytest tests/unit/test_esp32_v2_pipeline.py -q\n

## 4. Optional re-training command\n
This reproduces the selected V2 final training recipe.

In [ ]:
# !python train_esp32_v2_expanded_local.py \
#   --msm-epochs 10 \
#   --head-epochs 18 \
#   --ft-epochs 0 \
#   --batch-size 128 \
#   --reservoir-size 64 \
#   --focal-gamma 1.2 \
#   --label-smoothing 0.01 \
#   --prototype-weight 0.0 \
#   --no-realworld \
#   --artifact-suffix useful11_final

## 5. ONNX validation

In [ ]:
import onnx\n
model = onnx.load(str(ONNX))\n
onnx.checker.check_model(model)\n
print('ONNX check OK:', ONNX)\n

## 6. Convert ONNX to TFLite FP32/INT8 in Colab\n
This cell requires `onnx2tf` and TensorFlow. The calibration array is already included as `exports/esp32_v2/calib_data_v2.npy`.

In [ ]:
from pathlib import Path\n
import shutil\n
\n
try:\n
    import onnx2tf\n
    import tensorflow as tf\n
except Exception as exc:\n
    raise RuntimeError('Install TensorFlow and onnx2tf in Colab/Python 3.10-3.11, restart runtime, then rerun this cell.') from exc\n
\n
TF_OUT = ROOT / 'exports' / 'esp32_v2' / 'tflite_useful11_final'\n
if TF_OUT.exists():\n
    shutil.rmtree(TF_OUT)\n
\n
onnx2tf.convert(\n
    input_onnx_file_path=str(ONNX),\n
    output_folder_path=str(TF_OUT),\n
    output_integer_quantized_tflite=True,\n
    quant_type='per-channel',\n
    custom_input_op_name_np_data_path=[[\n
        'input', str(CALIB), [0.0] * 6, [1.0] * 6\n
    ]],\n
)\n
print('TFLite output folder:', TF_OUT)\n

## 7. Convert TFLite to ESP32 C header

In [ ]:
from pathlib import Path\n
\n
def tflite_to_c_array(tflite_path, header_path, array_name='g_sensorfusion_v2_model'):\n
    data = Path(tflite_path).read_bytes()\n
    hex_bytes = ', '.join(f'0x{b:02x}' for b in data)\n
    lines = [\n
        '#pragma once',\n
        '#include <cstdint>',\n
        f'alignas(16) const unsigned char {array_name}[] = {{',\n
        hex_bytes,\n
        '};',\n
        f'const unsigned int {array_name}_len = {len(data)};',\n
    ]\n
    Path(header_path).write_text('\\n'.join(lines))\n
    print('Wrote', header_path, 'bytes=', len(data))\n
\n
# Example after conversion; adjust filename if onnx2tf creates a different name:\n
# int8_model = next((ROOT / 'exports' / 'esp32_v2' / 'tflite_useful11_final').rglob('*int8*.tflite'))\n
# tflite_to_c_array(int8_model, ROOT / 'exports' / 'esp32_v2' / 'sensorfusion_v2_useful11_int8_model.h')\n

## 8. ESP32 inference reminder\n
Use `esp32_v2_useful11_config.h` for labels and normalization constants. Window IMU samples to `(50, 6)`, normalize channel-wise, then pass the tensor to the TFLite model.